In [0]:
volume_path = "/Volumes/fin_prod/bronze/arquivos_parquet"

tabelas = [
    "categorias_despesa",
    "compras_insumos",
    "compras_mercadorias",
    "contas_pagar",
    "contas_receber",
    "fluxo_caixa",
    "fornecedores",
    "funcionarios",
    "gastos_empresa",
    "pagamento_funcionarios"
]

codec = "snappy"

print(f"Volume: {volume_path}")
print(f"Total de tabelas: {len(tabelas)}")
print(f"Codec: {codec}")

In [0]:
for tabela in tabelas:
    print(f"\n{'='*50}")
    print(f"Tabela: {tabela}")
    df = spark.read.table(f"fin_prod.bronze.{tabela}")
    print(f"Linhas: {df.count()}")
    df.printSchema()

In [0]:
def get_size_mb(path):
    try:
        files = dbutils.fs.ls(path)
        total = 0
        for f in files:
            if f.size > 0:
                total += f.size
            else:
                try:
                    total += sum(x.size for x in dbutils.fs.ls(f.path))
                except:
                    pass
        return round(total / (1024 * 1024), 2)
    except:
        return 0

print(f"\n{'Tabela':<30} {'Delta (MB)':>12} {'Parquet (MB)':>14} {'Diferença':>12}")
print("-" * 72)

for tabela in tabelas:
    try:
        detail   = spark.sql(f"DESCRIBE DETAIL fin_prod.bronze.{tabela}")
        delta_mb = round(detail.collect()[0]["sizeInBytes"] / (1024 * 1024), 2)
    except:
        delta_mb = 0

    parquet_mb = get_size_mb(f"{volume_path}/{codec}/{tabela}")
    diff       = round(parquet_mb - delta_mb, 2)
    sinal      = f"+{diff}" if diff > 0 else str(diff)

    print(f"{tabela:<30} {delta_mb:>12} {parquet_mb:>14} {sinal:>12} MB")

In [0]:
print(f"\n{'Tabela':<30} {'Delta':>8} {'Parquet':>10} {'OK?':>6}")
print("-" * 58)

for tabela in tabelas:
    try:
        delta_count   = spark.read.table(f"fin_prod.bronze.{tabela}").count()
        parquet_count = spark.read.parquet(f"{volume_path}/{codec}/{tabela}").count()
        ok = "✅" if delta_count == parquet_count else "❌"
        print(f"{tabela:<30} {delta_count:>8} {parquet_count:>10} {ok:>6}")
    except Exception as e:
        print(f"{tabela:<30}  ERRO: {e}")

In [0]:
from datetime import date
from pyspark.sql.functions import current_timestamp, lit

# prefixo dbfs: é obrigatório para escrita via Spark no Volume
volume_path = "dbfs:/Volumes/fin_prod/bronze/arquivos_parquet"
codec       = "snappy"
erros       = []

tabelas = [
    "categorias_despesa",
    "compras_insumos",
    "compras_mercadorias",
    "contas_pagar",
    "contas_receber",
    "fluxo_caixa",
    "fornecedores",
    "funcionarios",
    "gastos_empresa",
    "pagamento_funcionarios"
]

for tabela in tabelas:
    try:
        print(f"🔄 Salvando: {tabela}")

        df = spark.read.table(f"fin_prod.bronze.{tabela}")

        df_final = df \
            .withColumn("_ingestion_timestamp", current_timestamp()) \
            .withColumn("_partition_date", lit(str(date.today())))

        output_path = f"{volume_path}/{codec}/{tabela}"

        df_final.write \
            .mode("overwrite") \
            .option("compression", codec) \
            .parquet(output_path)

        # Confirma imediatamente
        count = spark.read.parquet(output_path).count()
        print(f"  ✅ {tabela} → {count} registros gravados")

    except Exception as e:
        erros.append(tabela)
        print(f"  ❌ Erro em {tabela}: {str(e)}")

print(f"\n{'='*50}")
print(f"Sucesso: {len(tabelas) - len(erros)}/{len(tabelas)}")
if erros:
    print(f"Erros:   {erros}")

In [0]:
volume_path = "dbfs:/Volumes/fin_prod/bronze/arquivos_parquet/snappy"

tabelas = [
    "categorias_despesa", "compras_insumos", "compras_mercadorias",
    "contas_pagar", "contas_receber", "fluxo_caixa",
    "fornecedores", "funcionarios", "gastos_empresa", "pagamento_funcionarios"
]

print(f"\n{'Tabela':<30} {'Parquet (KB)':>14} {'Delta (KB)':>12} {'Diferença':>12}")
print("-" * 72)

for tabela in tabelas:
    # Tamanho Parquet
    try:
        arquivos  = dbutils.fs.ls(f"{volume_path}/{tabela}")
        pq_bytes  = sum(f.size for f in arquivos if f.size > 0)
        pq_kb     = round(pq_bytes / 1024, 2)
    except:
        pq_kb = 0

    # Tamanho Delta
    try:
        detail   = spark.sql(f"DESCRIBE DETAIL fin_prod.bronze.{tabela}")
        dl_bytes = detail.collect()[0]["sizeInBytes"]
        dl_kb    = round(dl_bytes / 1024, 2)
    except:
        dl_kb = 0

    diff  = round(pq_kb - dl_kb, 2)
    sinal = f"+{diff}" if diff > 0 else str(diff)
    print(f"{tabela:<30} {pq_kb:>14} {dl_kb:>12} {sinal:>12} KB")